# JSON Ingestion for RAG

This notebook shows practical ingestion for:
- Nested JSON files
- JSON Lines (JSONL) event streams

Focus: produce metadata-rich `Document` objects and chunk them for retrieval.

In [ ]:
import json
from pathlib import Path
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter


PROJECT_ROOT = Path.cwd().resolve().parents[1]
DATA_DIR = PROJECT_ROOT / "data"
JSON_PATH = DATA_DIR / "company_data.json"
JSONL_PATH = DATA_DIR / "events.jsonl"

if not JSON_PATH.exists():
    raise FileNotFoundError(f"Missing JSON file: {JSON_PATH}")
if not JSONL_PATH.exists():
    raise FileNotFoundError(f"Missing JSONL file: {JSONL_PATH}")

print("JSON:", JSON_PATH)
print("JSONL:", JSONL_PATH)

In [ ]:
with open(JSON_PATH, "r", encoding="utf-8") as f:
    company_data = json.load(f)

employee_docs = []
for employee in company_data.get("employees", []):
    employee_docs.append(
        Document(
            page_content=json.dumps(employee, ensure_ascii=False, indent=2),
            metadata={
                "source": str(JSON_PATH),
                "record_type": "employee",
                "employee_id": employee.get("id"),
                "employee_name": employee.get("name"),
            },
        )
    )

print("Employee docs:", len(employee_docs))
print("First employee metadata:", employee_docs[0].metadata)

In [ ]:
event_docs = []
with open(JSONL_PATH, "r", encoding="utf-8") as f:
    for idx, line in enumerate(f, start=1):
        line = line.strip()
        if not line:
            continue
        event_obj = json.loads(line)
        event_docs.append(
            Document(
                page_content=json.dumps(event_obj, ensure_ascii=False),
                metadata={
                    "source": str(JSONL_PATH),
                    "record_type": "event",
                    "line_number": idx,
                    "event_name": event_obj.get("event"),
                },
            )
        )

print("Event docs:", len(event_docs))
print("First event:", event_docs[0].page_content)

In [ ]:
all_json_docs = employee_docs + event_docs

splitter = RecursiveCharacterTextSplitter(
    chunk_size=600,
    chunk_overlap=80,
)
json_chunks = splitter.split_documents(all_json_docs)

print("Total input docs:", len(all_json_docs))
print("Total chunks:", len(json_chunks))
print("First chunk metadata:", json_chunks[0].metadata)
print("First chunk preview:", json_chunks[0].page_content[:180], "...")